# 31: Gold shard export

Packs a release's images into MosaicML shards, a few large files per split that training streams from, under `gold/<dataset_version>/shards/`. Configured by the file in `config/gold/exports/` named by `EXPORT_NAME`.

Run after `30_create_gold_manifest` for the same release. The first cell installs `mosaicml-streaming` (pinned to `pyproject.toml`), so there's nothing to set up. Shards are a rebuildable cache: rerunning fully rewrites them, and nothing deletes them automatically ([ADR 006](../docs/decisions/006-gold-shard-retention-undecided.md)).

In [ ]:
%pip install --quiet mosaicml-streaming==0.13.0


In [ ]:
%run ./_setup_env


In [ ]:
from pathlib import Path
import tempfile

from data_platform.files import stage_archives_and_extract_metadata
from data_platform.dataset_layout import build_layout, load_dataset_config, load_yaml_config, resolve_archive_paths
from data_platform.shard_export import write_gold_shards_for_splits
from data_platform.spark_io import (
    export_source_metadata_csv,
    load_manifest_rows_for_export,
    print_export_outputs,
)

GOLD_TABLES = {"manifest_rows": "gold.manifest_rows"}

# Set by the build_release job's parameters; a manual run uses this default.
dbutils.widgets.text("release_name", "sample-v1")
EXPORT_NAME = dbutils.widgets.get("release_name")
EXPORT_CONFIG = load_yaml_config(CONFIG_ROOT / "gold" / "exports" / f"{EXPORT_NAME}.yaml")

DATASET_VERSION = EXPORT_CONFIG["dataset_version"]
SHARD_SIZE_LIMIT_BYTES = EXPORT_CONFIG["shard_size_limit_bytes"]

STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
GOLD_ROOT = f"{STORAGE_CONFIG['storage_root'].rstrip('/')}/gold"
EXPORT_SHARD_ROOT = f"{GOLD_ROOT}/{DATASET_VERSION}/shards"

print(f"Exporting dataset_version={DATASET_VERSION!r} to {EXPORT_SHARD_ROOT}")


## Look up the release

Fails if notebook 30 hasn't built this release yet.

In [ ]:
registry_rows = spark.sql(
    f"SELECT * FROM gold.manifest_registry WHERE dataset_version = '{DATASET_VERSION}'"
).collect()
if not registry_rows:
    raise ValueError(
        f"No gold.manifest_registry row for dataset_version={DATASET_VERSION!r} -- "
        "run 30_create_gold_manifest.ipynb for this release first."
    )
registry_row = registry_rows[0]

DATASET_KEYS = registry_row["dataset_keys"]
preprocessing_versions = registry_row["preprocessing_versions"]
if len(preprocessing_versions) != 1:
    raise ValueError(
        f"dataset_version={DATASET_VERSION!r} spans preprocessing_versions={preprocessing_versions!r} "
        "-- shard export isn't designed for a mixed-preprocessing release."
    )
PREPROCESSING_VERSION = preprocessing_versions[0]

print(f"dataset_keys: {DATASET_KEYS}")
print(f"preprocessing_version: {PREPROCESSING_VERSION}")


## Stage the source archives locally

In [ ]:
staged_path_by_archive_uri = {}
bronze_tables_by_dataset_key = {}

for dataset_key in DATASET_KEYS:
    dataset_config = load_dataset_config(CONFIG_ROOT / "bronze" / "datasets" / f"{dataset_key}.yaml")
    dataset_config["landing_root"] = STORAGE_CONFIG["landing_root"]
    dataset_config["storage_root"] = STORAGE_CONFIG["storage_root"]
    dataset_layout = build_layout(dataset_config)
    bronze_tables_by_dataset_key[dataset_key] = dataset_layout["bronze_tables"]
    archives = resolve_archive_paths(
        dataset_config["archives"], dataset_layout["landing_paths"], dataset_layout["bronze_paths"]
    )
    local_stage_root = Path(tempfile.gettempdir()) / dataset_key / "archive_stage"
    stage_archives_and_extract_metadata(archives, local_stage_root)
    for archive in archives:
        staged_path_by_archive_uri[archive["archive_dbfs_path"]] = archive["staged_archive_path"]

print(f"Staged {len(staged_path_by_archive_uri)} archive(s) across {len(DATASET_KEYS)} dataset(s)")


## Load the release's rows

In [ ]:
rows_by_split = load_manifest_rows_for_export(spark, GOLD_TABLES, DATASET_VERSION)

for split, rows in rows_by_split.items():
    print(f"{split}: {len(rows)} manifest rows")


## Write the shards

Streams each archive once and re-checks every image's checksum against the manifest; any mismatch fails the export ([ADR 005](../docs/decisions/005-immutable-source-archives-checksum-verified.md)).

In [ ]:
shard_dir_by_split = {split: f"{EXPORT_SHARD_ROOT}/{split}" for split in rows_by_split}

per_split_summaries = write_gold_shards_for_splits(
    rows_by_split, staged_path_by_archive_uri, shard_dir_by_split, SHARD_SIZE_LIMIT_BYTES
)

for split, summary in per_split_summaries.items():
    print(f"{split}: wrote {summary['sample_count']} samples to {summary['shard_dir']}")


## Export the metadata CSVs (optional)

If `export_metadata_csv: true`, writes each dataset's source metadata for this release's images, as input for future image-plus-metadata models ([ADR 009](../docs/decisions/009-pin-metadata-feature-config-per-training-run.md)).

In [ ]:
if EXPORT_CONFIG.get("export_metadata_csv", False):
    METADATA_EXPORT_ROOT = f"{GOLD_ROOT}/{DATASET_VERSION}/metadata"
    dbutils.fs.mkdirs(METADATA_EXPORT_ROOT)

    image_ids_by_dataset_key = {dataset_key: [] for dataset_key in DATASET_KEYS}
    for rows in rows_by_split.values():
        for row in rows:
            image_ids_by_dataset_key[row["dataset_key"]].append(row["image_id"])

    for dataset_key, image_ids in image_ids_by_dataset_key.items():
        destination_path = f"{METADATA_EXPORT_ROOT}/{dataset_key}_source_metadata.csv"
        row_count = export_source_metadata_csv(
            spark, bronze_tables_by_dataset_key[dataset_key]["source_metadata"], dataset_key, image_ids, destination_path
        )
        print(f"{dataset_key}: wrote {row_count} metadata rows to {destination_path}")
else:
    print("export_metadata_csv not set in this export config -- skipping metadata export")


## Review

Fails if any split's sample count doesn't match the manifest.

In [ ]:
print_export_outputs(spark, display, DATASET_VERSION, per_split_summaries)
